# GLEE Competition agent V2

An adaptive, score-oriented agent built against the official `llms.txt` schemas. V2 uses configuration-aware game theory, within-game and disclosed-opponent learning, Bayesian persuasion, and a validated fallback around every move.

Run the cells in order. Keep only one Play cell running for an API key at a time.


In [ ]:
%pip install -q -U glee-sdk


In [ ]:
import hashlib
import math
import os
import threading
from collections import defaultdict, deque
from getpass import getpass

os.environ["GLEE_API_KEY"] = getpass("GLEE API key: ")

MODEL_LOCK = threading.Lock()
BARGAINING_MODELS = defaultdict(lambda: {"rejected_share": 0.0})
SELLER_MODELS = defaultdict(lambda: {"high_yes": 2.0, "high_total": 3.0,
                                            "low_yes": 1.0, "low_total": 3.0})
SEEN_EVENTS = set()
DECISION_LOG = deque(maxlen=500)


## Shared helpers and opponent memory

Disclosed opponents receive persistent profiles across games. Hidden opponents receive a game-local profile. Stable hashing provides reproducible mixed strategies without global random-state races.


In [ ]:
def clamp(value, low, high):
    return max(low, min(high, value))

def progress(state):
    round_no = max(1, int(state.get("round", 1)))
    maximum = state.get("max_rounds")
    if state.get("horizon_known") and maximum:
        return clamp((round_no - 1) / max(1, int(maximum) - 1), 0.0, 1.0)
    return min(0.8, (round_no - 1) / 10)

def is_final_round(state):
    return bool(state.get("horizon_known") and state.get("max_rounds") and
                int(state.get("round", 1)) >= int(state["max_rounds"]))

def stable_unit(game, salt=""):
    token = f"{game.get('game_id', '')}:{game['game_state'].get('round', 1)}:{salt}"
    number = int.from_bytes(hashlib.sha256(token.encode()).digest()[:8], "big")
    return number / 2**64

def opponent_key(game):
    opponent = game.get("opponent") or {}
    if opponent.get("type") != "hidden" and opponent.get("name"):
        return f"named:{opponent['name']}"
    return f"game:{game.get('game_id', '')}"

def player_index(player):
    return 1 if player in {"player_1", "alice"} else 2

def other_player(player):
    return "player_2" if player_index(player) == 1 else "player_1"

def allocation(offer, player):
    keys = (("player_1_gain", "alice_gain") if player_index(player) == 1 else
            ("player_2_gain", "bob_gain"))
    return next((float(offer[key]) for key in keys if key in offer), None)

def positive_signal(value):
    if isinstance(value, dict):
        value = value.get("decision", value.get("message"))
    if value is None:
        return None
    text = str(value).strip().lower()
    negatives = ("do not", "don't", "not worth", "avoid", "bad")
    if text in {"no", "not_recommended"} or any(x in text for x in negatives):
        return False
    if text in {"yes", "recommend", "recommended"}:
        return True
    positives = ("recommend", "worth it", "great", "strong product", "buy")
    return True if any(x in text for x in positives) else None

def logistic(value):
    return 1 / (1 + math.exp(-clamp(value, -60, 60)))


## 1. Bargaining: acceptance-probability optimization

V2 estimates the responder's minimum share from rejected offers, combines it with the visible discount structure, and searches candidate splits for the largest probability-weighted personal payoff. Acceptance compares the current discounted gain with estimated continuation value.


In [ ]:
def player_delta(state, player, default=0.93):
    value = state.get(f"delta_{player_index(player)}", default)
    return clamp(float(value), 0.01, 0.999)

def equilibrium_responder_share(proposer_delta, responder_delta):
    denominator = 1 - proposer_delta * responder_delta
    proposer_share = ((1 - responder_delta) / denominator
                      if denominator > 1e-9 else 0.5)
    return clamp(1 - proposer_share, 0.05, 0.95)

def update_bargaining_model(game):
    state = game["game_state"]
    me = game.get("your_player", state["current_player"])
    money = float(state["money_to_divide"])
    key = opponent_key(game)
    with MODEL_LOCK:
        model = BARGAINING_MODELS[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            offer = record.get("offer") or {}
            proposer = record.get("proposer", offer.get("proposer"))
            decision = record.get("decision")
            if isinstance(decision, dict):
                decision = decision.get("decision")
            event = (game.get("game_id"), record.get("round"), proposer, str(decision))
            if event in SEEN_EVENTS:
                continue
            SEEN_EVENTS.add(event)
            if proposer == me and str(decision).lower() == "reject":
                opponent_gain = allocation(offer, other_player(me))
                if opponent_gain is not None and money > 0:
                    model["rejected_share"] = max(model["rejected_share"],
                                                          opponent_gain / money)
        return dict(model)

def bargaining_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    me = game.get("your_player", state["current_player"])
    opponent = other_player(me)
    money = float(state["money_to_divide"])
    t = progress(state)
    my_delta = player_delta(state, me)
    opponent_delta = player_delta(state, opponent)
    model = update_bargaining_model(game)

    if state.get("complete_information"):
        prior_threshold = equilibrium_responder_share(my_delta, opponent_delta)
        prior_threshold = min(0.48, prior_threshold)
    else:
        prior_threshold = 0.38 + 0.06 * t
    threshold = clamp(max(prior_threshold, model["rejected_share"] + 0.01), 0.20, 0.65)

    if action_type == "offer":
        best = None
        for step in range(20, 66):
            responder_share = step / 100
            # At the estimated reservation share, acceptance should already be likely.
            accept_probability = logistic((responder_share - threshold + 0.04) / 0.02)
            own_share = 1 - responder_share
            # Slightly convex utility rewards high-percentile payoffs without ignoring deal risk.
            objective = accept_probability * own_share**1.25
            candidate = (objective, own_share)
            if best is None or candidate > best:
                best = candidate
        own_share = (1 - 0.25 * t) * best[1] + 0.25 * t * 0.5
        own_gain = round(money * clamp(own_share, 0.35, 0.80), 8)
        other_gain = money - own_gain
        if player_index(me) == 1:
            action = {"alice_gain": own_gain, "bob_gain": other_gain}
        else:
            action = {"alice_gain": other_gain, "bob_gain": own_gain}
        if state.get("messages_allowed"):
            action["message"] = "Delay reduces value; this split settles efficiently now."
        return action

    current_gain = allocation(state.get("last_offer") or {}, me)
    if current_gain is None:
        return {"decision": "reject"}
    if is_final_round(state):
        return {"decision": "accept" if current_gain >= 0 else "reject"}
    next_accept_probability = 0.72 + 0.20 * t
    continuation = money * (1 - threshold) * my_delta * next_accept_probability
    reservation_floor = money * (0.42 - 0.12 * t)
    required = max(reservation_floor, continuation)
    return {"decision": "accept" if current_gain >= required else "reject"}


## 2. Negotiation: asymmetric surplus capture

With complete information, V2 claims 85% of the available surplus early and concedes toward 60%. With hidden values, it anchors on observed opponent prices instead of relying only on fixed valuation multipliers.


In [ ]:
def negotiation_prices(state):
    prices = []
    for record in state.get("history", []):
        if not isinstance(record, dict):
            continue
        for field in ("offer", "counteroffer"):
            item = record.get(field)
            if isinstance(item, dict) and item.get("price") is not None:
                prices.append((item.get("from_player", record.get("decided_by")),
                               float(item["price"])))
    return prices

def negotiation_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    me = game.get("your_player", state["current_player"])
    opponent = other_player(me)
    role = state[f"{me}_role"]
    my_value = float(state[f"{me}_value"])
    t = progress(state)
    opponent_value = state.get(f"{opponent}_value")

    if state.get("complete_information") and opponent_value is not None:
        opponent_value = float(opponent_value)
        seller_value = my_value if role == "seller" else opponent_value
        buyer_value = my_value if role == "buyer" else opponent_value
        surplus = buyer_value - seller_value
        if surplus <= 0:
            target = my_value
        else:
            capture = 0.85 - 0.25 * t
            target = (seller_value + capture * surplus if role == "seller" else
                      seller_value + (1 - capture) * surplus)
    else:
        observed = [price for player, price in negotiation_prices(state) if player == opponent]
        if observed and role == "seller":
            target = max(my_value, max(observed) * (1.18 - 0.13 * t))
        elif observed:
            target = min(my_value, min(observed) * (0.82 + 0.13 * t))
        elif role == "seller":
            target = my_value * (1.60 - 0.25 * t)
        else:
            target = my_value * (0.60 + 0.30 * t)

    if action_type == "offer":
        action = {"product_price": round(target, 8)}
        if state.get("messages_allowed"):
            action["message"] = "This gives both sides a positive reason to close now."
        return action

    price = float((state.get("last_offer") or {})["price"])
    profitable = price >= my_value if role == "seller" else price <= my_value
    if is_final_round(state):
        return {"decision": "AcceptOffer" if profitable else "RejectOffer"}
    target_utility = ((target - my_value) if role == "seller" else (my_value - target))
    offered_utility = ((price - my_value) if role == "seller" else (my_value - price))
    required_fraction = 0.90 - 0.25 * t
    if profitable and offered_utility >= max(0.0, required_fraction * target_utility):
        return {"decision": "AcceptOffer"}
    counter = (1 - 0.45 * t) * target + 0.45 * t * price
    counter = max(my_value, counter) if role == "seller" else min(my_value, counter)
    action = {"decision": "RejectOffer", "product_price": round(counter, 8)}
    if state.get("messages_allowed"):
        action["message"] = "I can move, but not beyond a profitable agreement."
    return action


## 3. Persuasion: Bayesian trust and finite-horizon pooling

The buyer maintains Beta-smoothed reliability estimates, including across disclosed opponents. The seller protects reputation early and approaches the Bayesian-persuasion pooling rate near the end.


In [ ]:
def update_seller_model(game):
    state = game["game_state"]
    key = opponent_key(game)
    with MODEL_LOCK:
        model = SELLER_MODELS[key]
        for record in state.get("history", []):
            if not isinstance(record, dict):
                continue
            quality = record.get("quality")
            signal = positive_signal(record.get("seller_message"))
            event = (game.get("game_id"), "persuasion", record.get("round"))
            if event in SEEN_EVENTS or quality not in {"high", "low"} or signal is None:
                continue
            SEEN_EVENTS.add(event)
            model[f"{quality}_total"] += 1
            model[f"{quality}_yes"] += float(signal)
        return dict(model)

def persuasion_strategy(game):
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    price = float(state["product_price"])
    round_no = max(1, int(state.get("round", 1)))
    total_rounds = max(round_no, int(state.get("total_rounds", round_no)))
    t = (round_no - 1) / max(1, total_rounds - 1)

    if action_type in {"seller_message", "seller_recommendation"}:
        quality = state.get("current_quality")
        recommend = quality == "high"
        if quality == "low" and "v" in state and "u" in state:
            p, v, u = float(state["p"]), float(state["v"]), float(state["u"])
            if price <= u:
                recommend = True
            elif u < price < v and 0 < p < 1:
                cutoff = (price - u) / (v - u)
                static_pool = p * (1 - cutoff) / (cutoff * (1 - p))
                reputation_weight = 0.15 + 0.85 * t**1.5
                pool_probability = clamp(static_pool * reputation_weight, 0.0, 1.0)
                recommend = stable_unit(game, "seller-pool") < pool_probability
        if action_type == "seller_recommendation":
            return {"decision": "yes" if recommend else "no"}
        return {"message": ("I recommend buying this product." if recommend else
                            "I do not recommend buying this product this round.")}

    p, v, u = float(state["p"]), float(state["v"]), float(state["u"])
    model = update_seller_model(game)
    q_high = model["high_yes"] / model["high_total"]
    q_low = model["low_yes"] / model["low_total"]
    signal = positive_signal(state.get("seller_message"))
    if signal is None:
        posterior = p
    else:
        likelihood_high = q_high if signal else 1 - q_high
        likelihood_low = q_low if signal else 1 - q_low
        denominator = p * likelihood_high + (1 - p) * likelihood_low
        posterior = p * likelihood_high / denominator if denominator > 1e-12 else p
    expected_value = posterior * v + (1 - posterior) * u
    remaining = total_rounds - round_no
    uncertainty = 1 / math.sqrt(model["high_total"] + model["low_total"])
    information_bonus = (0.025 * remaining * (v - u) * uncertainty
                         if signal is True else 0.0)
    return {"decision": "yes" if expected_value + information_bonus >= price else "no"}


## Validated dispatcher and safe fallbacks

Every proposed action is checked before submission. Any strategy exception or malformed result becomes a conservative legal action, preventing invalid-move and timeout penalties. Inspect `DECISION_LOG` when tuning.


In [ ]:
STRATEGIES = {
    "bargaining": bargaining_strategy,
    "negotiation": negotiation_strategy,
    "persuasion": persuasion_strategy,
}

def fallback_action(game):
    family = game["game_family"]
    state = game["game_state"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining":
        if action_type == "offer":
            money = float(state["money_to_divide"])
            alice = round(money / 2, 8)
            return {"alice_gain": alice, "bob_gain": money - alice}
        return {"decision": "accept"}
    if family == "negotiation":
        me = game.get("your_player", state["current_player"])
        role = state[f"{me}_role"]
        value = float(state[f"{me}_value"])
        if action_type == "offer":
            return {"product_price": value}
        price = float((state.get("last_offer") or {}).get("price", value))
        profitable = price >= value if role == "seller" else price <= value
        if profitable:
            return {"decision": "AcceptOffer"}
        if is_final_round(state):
            return {"decision": "RejectOffer"}
        return {"decision": "RejectOffer", "product_price": value}
    if action_type == "seller_message":
        return {"message": "I recommend this product."}
    if action_type == "seller_recommendation":
        return {"decision": "yes"}
    expected = float(state["p"]) * float(state["v"]) + (1 - float(state["p"])) * float(state["u"])
    return {"decision": "yes" if expected >= float(state["product_price"]) else "no"}

def validate_action(game, action):
    if not isinstance(action, dict):
        raise ValueError("strategy did not return a dictionary")
    family = game["game_family"]
    action_type = game["valid_actions"]["type"]
    if family == "bargaining" and action_type == "offer":
        if "alice_gain" not in action or "bob_gain" not in action:
            raise ValueError("bargaining offer lacks Alice/Bob gains")
        total = float(action["alice_gain"]) + float(action["bob_gain"])
        if not math.isclose(total, float(game["game_state"]["money_to_divide"]), abs_tol=1e-7):
            raise ValueError("bargaining gains do not sum to the pot")
    elif family == "bargaining":
        if action.get("decision") not in {"accept", "reject", "walkaway"}:
            raise ValueError("invalid bargaining decision")
    elif family == "negotiation" and action_type == "offer":
        float(action["product_price"])
    elif family == "negotiation":
        if action.get("decision") not in {"AcceptOffer", "RejectOffer", "WalkAway"}:
            raise ValueError("invalid negotiation decision")
        if action.get("decision") == "RejectOffer" and not is_final_round(game["game_state"]):
            float(action["product_price"])
    elif action_type == "seller_message":
        if not isinstance(action.get("message"), str) or len(action["message"]) > 2000:
            raise ValueError("invalid seller message")
    else:
        if action.get("decision") not in {"yes", "no"}:
            raise ValueError("invalid persuasion decision")
    return action

def strategy(game):
    error = None
    try:
        action = validate_action(game, STRATEGIES[game["game_family"]](game))
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        action = validate_action(game, fallback_action(game))
        print(f"SAFE FALLBACK {game.get('game_id')}: {error}")
    with MODEL_LOCK:
        DECISION_LOG.append({"game_id": game.get("game_id"),
                             "family": game.get("game_family"),
                             "round": game.get("game_state", {}).get("round"),
                             "action": dict(action), "error": error})
    return action


## Offline schema tests

These tests exercise the exact bargaining distinction from `llms.txt`: outgoing gains use Alice/Bob names, while `last_offer` uses player-number names. No API calls are made.


In [ ]:
def run_smoke_tests():
    bargaining_offer = {
        "game_id": "test-b", "game_family": "bargaining", "your_player": "player_1",
        "opponent": {"type": "hidden", "name": None},
        "valid_actions": {"type": "offer", "fields": {}},
        "game_state": {"current_player": "player_1", "round": 1, "max_rounds": 5,
                       "horizon_known": True, "money_to_divide": 100, "delta_1": .9,
                       "delta_2": .9, "complete_information": True, "history": []}
    }
    action = strategy(bargaining_offer)
    assert set(action) >= {"alice_gain", "bob_gain"}
    assert math.isclose(action["alice_gain"] + action["bob_gain"], 100)

    bargaining_decision = dict(bargaining_offer)
    bargaining_decision["your_player"] = "player_2"
    bargaining_decision["valid_actions"] = {"type": "decision", "fields": {}}
    bargaining_decision["game_state"] = dict(bargaining_offer["game_state"],
        current_player="player_2", last_offer={"player_1_gain": 54, "player_2_gain": 46})
    assert strategy(bargaining_decision)["decision"] in {"accept", "reject"}

    negotiation = {
        "game_id": "test-n", "game_family": "negotiation", "your_player": "player_1",
        "opponent": {"type": "hidden", "name": None},
        "valid_actions": {"type": "offer", "fields": {}},
        "game_state": {"current_player": "player_1", "player_1_role": "seller",
                       "player_2_role": "buyer", "player_1_value": 40,
                       "player_2_value": 100, "complete_information": True,
                       "round": 1, "max_rounds": 5, "horizon_known": True, "history": []}
    }
    assert 40 <= strategy(negotiation)["product_price"] <= 100

    persuasion = {
        "game_id": "test-p", "game_family": "persuasion", "your_player": "player_1",
        "opponent": {"type": "hidden", "name": None},
        "valid_actions": {"type": "seller_recommendation", "fields": {}},
        "game_state": {"current_quality": "high", "product_price": 50, "p": .5,
                       "v": 100, "u": 0, "round": 1, "total_rounds": 5, "history": []}
    }
    assert strategy(persuasion) == {"decision": "yes"}
    print("All V2 smoke tests passed.")

run_smoke_tests()


## Play — run this cell only once

Persuasion is selected initially because the current agent has only 14 persuasion games. Change `GAME_FAMILIES` after it reaches a useful sample. Start with concurrency 1; increase only after a clean run with no fallbacks or API race warnings.


In [ ]:
from glee_sdk import GleeClient

GAME_FAMILIES = ["persuasion"]
MAX_GAMES = 86

client = GleeClient(api_key=os.environ["GLEE_API_KEY"])
client.run(strategy, game_families=GAME_FAMILIES, concurrency=1, max_games=MAX_GAMES)


## Inspect results


In [ ]:
print(client.stats())
print("Recent decisions:")
list(DECISION_LOG)[-10:]
